# Invoice OCR — Kaggle GPU
Dùng kernel GPU và bật Internet. Notebook chạy backend Transformers, nhận đầy đủ ảnh của mỗi hóa đơn.
Ở local, xuất ZIP bằng `invoice-ocr label export --split valid --zip`. ZIP cũ trước refactor phải xuất lại.


## Chuẩn bị code và nhãn
Chọn SPLIT khớp bộ reviewed đã xuất. Chỉ dùng valid để chỉnh prompt, giữ test cho đánh giá cuối.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, shutil, subprocess, sys, zipfile
SPLIT = 'valid'
RUN_NAME = 'transformers-' + SPLIT + '-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
PROJECT = Path('/kaggle/working/invoice-ocr')
DATASET = None  # Or Path('/kaggle/input/your-dataset')
input_root = Path(DATASET) if DATASET is not None else Path('/kaggle/input')
roots = sorted({p.parent for p in input_root.rglob('pyproject.toml')
                if (p.parent / f'groundtruth/reviewed/{SPLIT}.jsonl').is_file()})
if roots:
    assert len(roots) == 1, 'Select one dataset with DATASET.'
    shutil.copytree(roots[0], PROJECT, dirs_exist_ok=True)
else:
    archives = sorted(input_root.rglob(f'invoice-reviewed-{SPLIT}.zip'))
    assert len(archives) == 1, 'Upload one new-format reviewed ZIP.'
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in archive.namelist():
            assert '\\' not in name and (PROJECT/name).resolve().is_relative_to(PROJECT.resolve())
        archive.extractall(PROJECT)
OUTPUT = Path('/kaggle/working') / RUN_NAME


## Cài package và chuẩn bị GPU
CLI, schema và prompt dùng chung với app local. Cấu hình nằm trong `.env`; không sửa nhãn.


In [ ]:
for relative in ('pyproject.toml', 'src/invoice_ocr/cli.py', 'requirements/gpu.txt', 'configs/transformers.env.example'):
    assert (PROJECT/relative).is_file(), f'Missing {relative}; export a new ZIP after refactor.'
INPUT = PROJECT / f'groundtruth/reviewed/{SPLIT}.jsonl'
assert INPUT.is_file() and INPUT.read_text().strip(), 'Reviewed labels are required.'
ENV_FILE = PROJECT / '.env'
if not ENV_FILE.exists():
    shutil.copyfile(PROJECT/'configs/transformers.env.example', ENV_FILE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT)], check=True)
def run(*args):
    command = [sys.executable, '-m', 'invoice_ocr', *map(str, args)]
    if args[0] in ('gpu', 'extract', 'plan', 'db'):
        command += ['--env-file', str(ENV_FILE)]
    subprocess.run(command, cwd=PROJECT, check=True)
run('gpu', 'setup')
run('gpu', 'doctor')
run('gpu', 'download')


## Chạy trích xuất
OUTPUT mới cho từng lần chạy. Thêm `--limit 1` nếu muốn smoke test trước.


In [ ]:
run('plan', '--input', INPUT, '--output', OUTPUT/'raw')
run('extract', '--backend', 'transformers', '--device', 'auto',
    '--input', INPUT, '--output', OUTPUT/'raw')


## Đánh giá và xuất kết quả
Tải ZIP trước khi runtime kết thúc.


In [ ]:
run('evaluate', '--groundtruth', INPUT, '--predictions', OUTPUT/'raw', '--output', OUTPUT/'evaluation')
shutil.copyfile(INPUT, OUTPUT/'labels.jsonl')
print(shutil.make_archive(str(OUTPUT)+'-results', 'zip', root_dir=OUTPUT))
